In [11]:
# Challenge 3 — Imports (only what this challenge needs)
import logging
import os
import re
import unittest
from typing import Any, Dict
from unittest import mock

import requests

import google.auth
from google import genai
from google.adk.agents import LlmAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.tools import agent_tool, google_search
from google.genai import types

print("Imports ready.")

Imports ready.


In [12]:
def get_secret(name: str, prompt: str) -> str:
    """Read a credential from the environment, else prompt via input().

    getpass() hangs on some VS Code kernels, so we use input(). Nothing is
    hardcoded; set the env var beforehand to skip the prompt.
    """
    value = os.environ.get(name, "").strip()
    if value:
        print("{}: loaded from environment ({} chars).".format(name, len(value)))
        return value
    value = input(prompt).strip()
    os.environ[name] = value
    return value


GOOGLE_MAPS_API_KEY = get_secret(
    "GOOGLE_MAPS_API_KEY", "Enter your Google Maps Geocoding API key: ")
assert GOOGLE_MAPS_API_KEY, "Google Maps API key is required for geocoding."


def redact_secrets(text: str) -> str:
    """Strip ?key=... out of text before it can reach a notebook output.

    requests puts the full URL in its exception text. Also use Clear All Outputs.
    """
    if not text:
        return text
    return re.sub(r"([?&]key=)[^&\s]+", r"\1***REDACTED***", str(text))


# Model auth: Vertex AI via this kernel's ambient lab credentials.
_creds, _adc_project = google.auth.default()
GOOGLE_CLOUD_PROJECT = os.environ.get("GOOGLE_CLOUD_PROJECT", "").strip() or _adc_project
GOOGLE_CLOUD_LOCATION = os.environ.get("GOOGLE_CLOUD_LOCATION", "").strip() or "us-central1"

# google-genai only accepts "1"/"true" here and defaults to "0"; leaving it unset
# sends the client down the API-key path and raises "No API key was provided."
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "1"
os.environ["GOOGLE_CLOUD_PROJECT"] = GOOGLE_CLOUD_PROJECT
os.environ["GOOGLE_CLOUD_LOCATION"] = GOOGLE_CLOUD_LOCATION
# A Maps key here would shadow Vertex auth -> 403 API_KEY_SERVICE_BLOCKED.
os.environ.pop("GOOGLE_API_KEY", None)
os.environ.pop("GEMINI_API_KEY", None)

_probe = genai.Client()
assert _probe.vertexai, "Vertex mode did not engage; check the env vars above."
print("Vertex AI engaged: project={} location={}".format(
    GOOGLE_CLOUD_PROJECT, GOOGLE_CLOUD_LOCATION))

GOOGLE_MAPS_API_KEY: loaded from environment (39 chars).
Vertex AI engaged: project=qwiklabs-gcp-03-aa9fafb9374b location=us-central1


In [13]:
logger = logging.getLogger("multi_agent")
logger.setLevel(logging.INFO)
if not logger.handlers:            # re-running this cell must not duplicate handlers
    _h = logging.StreamHandler()
    _h.setFormatter(logging.Formatter("%(levelname)-7s %(name)s: %(message)s"))
    logger.addHandler(_h)
logger.propagate = False

MODEL = "gemini-2.5-flash"

NWS_API_BASE = "https://api.weather.gov"
# The NWS API mandates a User-Agent header; omitting it returns 403 Forbidden.
NWS_USER_AGENT = "(challenge3-multi-agent, nathaniel.morrow@example.com)"
GEOCODE_API_URL = "https://maps.googleapis.com/maps/api/geocode/json"
REQUEST_TIMEOUT_SECONDS = 15

APP_NAME = "multi_agent_app"
USER_ID = "workshop-user"
print("Configured.")

Configured.


In [14]:
def get_location_lat_long(city: str, state: str) -> Dict[str, Any]:
    """Convert a US city and state into latitude and longitude coordinates.

    Call this FIRST when the user names a place, then pass the coordinates to
    `get_current_weather`. Locations outside the US return an error.

    Args:
        city: The city name, for example "Denver".
        state: State name or two-letter abbreviation, for example "CO".

    Returns:
        {"status": "success", "formatted_address", "latitude", "longitude"} or
        {"status": "error", "error_message"}.
    """
    params = {"address": "{}, {}".format(city, state), "key": GOOGLE_MAPS_API_KEY}
    try:
        response = requests.get(GEOCODE_API_URL, params=params,
                                timeout=REQUEST_TIMEOUT_SECONDS)
        response.raise_for_status()
        payload = response.json()
    except requests.exceptions.RequestException as exc:
        # requests puts the full request URL in its exception text, and our key
        # rides along as a ?key= parameter -- redact before it reaches output.
        return {"status": "error",
                "error_message": redact_secrets("Geocoding failed: {}".format(exc))}

    results = payload.get("results") or []
    if payload.get("status") != "OK" or not results:
        return {"status": "error",
                "error_message": "Could not geocode '{}, {}' (status: {}).".format(
                    city, state, payload.get("status"))}

    top = results[0]
    country = next((c.get("short_name") for c in top.get("address_components", [])
                    if "country" in c.get("types", [])), None)
    if country and country != "US":
        return {"status": "error",
                "error_message": ("'{}' resolved to {}, outside the United States. "
                                  "The NWS only covers the US.".format(
                                      top.get("formatted_address"), country))}

    loc = top["geometry"]["location"]
    return {"status": "success",
            "formatted_address": top.get("formatted_address"),
            "latitude": loc["lat"], "longitude": loc["lng"]}


def get_current_weather(latitude: float, longitude: float) -> Dict[str, Any]:
    """Get the real-time NWS forecast for US coordinates.

    Needs coordinates, so call `get_location_lat_long` first if given a city name.
    Two-hop NWS flow: /points/{lat},{lon} returns metadata containing a generated
    `properties.forecast` URL; GET that for `properties.periods[]`, index 0 being
    the current period. A 404 means the point is outside US coverage.

    Args:
        latitude: Latitude in decimal degrees, for example 39.7392.
        longitude: Longitude in decimal degrees, for example -104.9903.

    Returns:
        {"status": "success", "period", "temperature", "temperature_unit",
        "short_forecast", "summary"} or {"status": "error", "error_message"}.
    """
    headers = {"User-Agent": NWS_USER_AGENT, "Accept": "application/geo+json"}
    try:
        points = requests.get("{}/points/{},{}".format(NWS_API_BASE, latitude, longitude),
                              headers=headers, timeout=REQUEST_TIMEOUT_SECONDS)
        if points.status_code == 404:
            return {"status": "error",
                    "error_message": ("No NWS data for {},{} -- most likely outside "
                                      "the United States.".format(latitude, longitude))}
        points.raise_for_status()
        forecast = requests.get(points.json()["properties"]["forecast"],
                                headers=headers, timeout=REQUEST_TIMEOUT_SECONDS)
        forecast.raise_for_status()
        periods = forecast.json()["properties"]["periods"]
    except requests.exceptions.RequestException as exc:
        return {"status": "error", "error_message": "NWS request failed: {}".format(exc)}
    except (KeyError, IndexError, ValueError) as exc:
        return {"status": "error", "error_message": "Unexpected NWS response: {}".format(exc)}

    if not periods:
        return {"status": "error", "error_message": "NWS returned an empty forecast."}

    now = periods[0]
    return {"status": "success", "period": now.get("name"),
            "temperature": now.get("temperature"),
            "temperature_unit": now.get("temperatureUnit"),
            "short_forecast": now.get("shortForecast"),
            "summary": "{}: {} Near {}\u00b0{}. {}".format(
                now.get("name"), now.get("shortForecast"), now.get("temperature"),
                now.get("temperatureUnit"), now.get("detailedForecast"))}


# Direct proof the tools chain before any agent is involved.
_loc = get_location_lat_long("Denver", "CO")
print(get_current_weather(_loc["latitude"], _loc["longitude"])["summary"])

This Afternoon: Chance Showers And Thunderstorms Near 90°F. A chance of showers and thunderstorms. Mostly sunny. High near 90, with temperatures falling to around 88 in the afternoon. East northeast wind around 9 mph. Chance of precipitation is 30%.


In [15]:
# Specialist 1 of 2. Same two-tool chaining design as Challenge 2: this agent
# decides to geocode, then decides to fetch the forecast.
#
# `description` matters more than usual here. Once this agent is wrapped as a
# tool, the ROOT agent's LLM reads this description to decide whether to route
# a question here, so it doubles as the routing contract.
weather_agent = LlmAgent(
    name="weather_agent",
    model=MODEL,
    description=(
        "Reports real-time US weather forecasts from the National Weather "
        "Service. Use for any question about current or upcoming weather, "
        "temperature, or forecasts in a US location."
    ),
    instruction=(
        "You are a US weather specialist.\n"
        "1. Call `get_location_lat_long` with the city and state for coordinates.\n"
        "2. Pass those coordinates to `get_current_weather`.\n"
        "Then summarize the forecast in two or three sentences: conditions, "
        "temperature, and any notable detail such as rain chance or wind.\n"
        "If a tool returns status 'error', report the problem plainly and never "
        "invent weather data. You cover only the US and its territories."
    ),
    tools=[get_location_lat_long, get_current_weather],
)
print("weather_agent ready with {} tools.".format(len(weather_agent.tools)))

weather_agent ready with 2 tools.


In [16]:
# Specialist 2 of 2. `google_search` is a built-in ADK tool.
#
# ADK constraint worth knowing: an agent using a built-in tool cannot also carry
# custom function tools. That is a second, structural reason search has to be
# its own agent rather than one more tool bolted onto the root agent.
search_agent = LlmAgent(
    name="search_agent",
    model=MODEL,
    description=(
        "Answers general knowledge questions by searching Google. Use for facts, "
        "news, people, places, definitions, and anything not about US weather."
    ),
    instruction=(
        "You are a research specialist. Use the `google_search` tool to find "
        "current, accurate information, then answer in two or three sentences. "
        "Cite the source site when it is useful. If the search returns nothing "
        "relevant, say so rather than guessing."
    ),
    tools=[google_search],
)
print("search_agent ready with {} tools.".format(len(search_agent.tools)))

search_agent ready with 1 tools.


In [17]:
# Agent 3: the ROOT agent. Every ADK system has exactly one.
#
# Both specialists are wrapped with AgentTool, so they are CALLED and RETURN --
# control comes back here and the root composes the final answer. That is what
# keeps the user talking only to the root agent.
#
# Contrast with sub_agents, which TRANSFERS control: the specialist becomes the
# responder and the conversation stays there. Not what we want, so sub_agents
# is left empty and the delegation happens entirely through tools.
ROOT_INSTRUCTION = (
    "You are the primary assistant. You do not answer questions from your own "
    "knowledge; you delegate to a specialist and relay the result.\n"
    "\n"
    "Routing rules:\n"
    "- US weather, forecasts, or temperature -> call `weather_agent`.\n"
    "- Any other factual question -> call `search_agent`.\n"
    "- Weather outside the US -> call `weather_agent` anyway and relay its "
    "explanation that the National Weather Service is US-only.\n"
    "- A question needing both, for example 'what is the weather where the "
    "Super Bowl is played this year', -> call `search_agent` for the location "
    "first, then `weather_agent` for the forecast.\n"
    "\n"
    "Present the specialist's answer in your own words as one clear reply. Name "
    "which specialist you consulted. Never fabricate an answer if a specialist "
    "fails -- say what went wrong."
)

root_agent = LlmAgent(
    name="root_agent",
    model=MODEL,
    description="Primary assistant that routes questions to specialist agents.",
    instruction=ROOT_INSTRUCTION,
    tools=[
        agent_tool.AgentTool(agent=weather_agent),
        agent_tool.AgentTool(agent=search_agent),
    ],
)

session_service = InMemorySessionService()
runner = Runner(app_name=APP_NAME, agent=root_agent, session_service=session_service)

print("root_agent ready. Specialists exposed as tools:")
for _tool in root_agent.tools:
    _delegate = getattr(_tool, "agent", None)
    if _delegate is not None:
        print("  - {}".format(_delegate.name))

root_agent ready. Specialists exposed as tools:
  - weather_agent
  - search_agent


In [18]:
async def ask(prompt: str, show_routing: bool = True) -> str:
    """Send one prompt to the root agent in a fresh session and return its reply.

    show_routing prints each delegation so the multi-agent hand-off is visible.
    """
    session = await session_service.create_session(app_name=APP_NAME, user_id=USER_ID)
    message = types.Content(role="user", parts=[types.Part(text=prompt)])
    final = "(no response)"

    async for event in runner.run_async(user_id=USER_ID, session_id=session.id,
                                        new_message=message):
        if show_routing:
            for call in event.get_function_calls():
                print("   -> delegating to {}".format(call.name))
            for resp in event.get_function_responses():
                print("   <- {} returned".format(resp.name))
        if event.is_final_response() and event.content and event.content.parts:
            texts = [p.text for p in event.content.parts if p.text]
            if texts:
                final = "".join(texts).strip()
    return final


# Top-level await works directly in notebook cells. Do NOT use asyncio.run() or
# nest_asyncio -- both break on ADK 2.x background tasks.
print("User: What's the weather in Denver, CO?\n")
print("Agent:", await ask("What's the weather in Denver, CO?"))

User: What's the weather in Denver, CO?

   -> delegating to weather_agent
   <- weather_agent returned
Agent: The weather specialist reports that in Denver, CO, this afternoon's forecast includes a chance of showers and thunderstorms, with a high temperature near 90°F. There is a 30% chance of precipitation, and winds will be from the east-northeast around 9 mph.


In [19]:
# Demonstration: each prompt should route to a different specialist. Watch the
# "-> delegating to" lines -- that is the multi-agent system doing its job.
DEMO = [
    ("Weather -> weather_agent",  "What's the weather in Miami, FL?"),
    ("Knowledge -> search_agent", "Who won the 2024 World Series?"),
    ("Weather -> weather_agent",  "How hot is it in Phoenix, AZ right now?"),
    ("Knowledge -> search_agent", "What is the tallest building in Chicago?"),
    ("Non-US weather",            "What's the weather in London, England?"),
    ("Needs both specialists",    "What's the weather in the city where the Gateway Arch is?"),
]

for label, prompt in DEMO:
    print("=" * 78)
    print("[{}]\nUser:  {}".format(label, prompt))
    answer = await ask(prompt)
    print("Agent: {}\n".format(answer))

[Weather -> weather_agent]
User:  What's the weather in Miami, FL?
   -> delegating to weather_agent
   <- weather_agent returned
Agent: According to the weather specialist, this afternoon in Miami, FL, there's a chance of showers and thunderstorms with a high near 89°F. The heat index could be as high as 106°F, and there's a 30% chance of precipitation.

[Knowledge -> search_agent]
User:  Who won the 2024 World Series?
   -> delegating to search_agent
   <- search_agent returned
Agent: The Los Angeles Dodgers won the 2024 World Series, defeating the New York Yankees in five games. This was their eighth World Series title, and Freddie Freeman was named the series MVP. This information was provided by the search agent.

[Weather -> weather_agent]
User:  How hot is it in Phoenix, AZ right now?
   -> delegating to weather_agent
   <- weather_agent returned
Agent: The weather specialist reports that it is currently sunny in Phoenix, AZ, with a high temperature near 113°F. The heat index co

In [20]:
# Tests: prove the multi-agent structure and the weather tools it depends on.
# Offline -- HTTP is mocked and no model is called, so this needs no API key.

def _fake_response(status_code=200, payload=None):
    resp = mock.Mock()
    resp.status_code = status_code
    resp.json.return_value = payload or {}
    if status_code >= 400:
        resp.raise_for_status.side_effect = requests.exceptions.HTTPError("boom")
    else:
        resp.raise_for_status.return_value = None
    return resp


def _tool_names(agent):
    """Tool identifiers for an agent, whether ADK kept callables or wrapped them."""
    return [getattr(t, "__name__", None) or getattr(t, "name", "") for t in agent.tools]


class TestMultiAgentStructure(unittest.TestCase):
    """Requirement: three agents, wired root -> two specialists."""

    def test_three_agents_exist(self):
        for agent, name in ((root_agent, "root_agent"),
                            (weather_agent, "weather_agent"),
                            (search_agent, "search_agent")):
            self.assertEqual(agent.name, name)

    def test_root_exposes_both_specialists_as_tools(self):
        delegates = [t.agent.name for t in root_agent.tools
                     if isinstance(t, agent_tool.AgentTool)]
        self.assertIn("weather_agent", delegates)
        self.assertIn("search_agent", delegates)
        self.assertEqual(len(delegates), 2)

    def test_specialists_are_tools_not_sub_agents(self):
        # AgentTool returns control to root, so the user only ever talks to root.
        # sub_agents would transfer control away and leave it there.
        self.assertFalse(root_agent.sub_agents)

    def test_specialists_have_descriptions_for_routing(self):
        # The root LLM routes off these descriptions, so they must be non-empty.
        for agent in (weather_agent, search_agent):
            self.assertTrue(agent.description.strip(), agent.name)

    def test_weather_agent_has_both_chaining_tools(self):
        names = _tool_names(weather_agent)
        self.assertIn("get_location_lat_long", names)
        self.assertIn("get_current_weather", names)

    def test_search_agent_uses_google_search(self):
        self.assertEqual(len(search_agent.tools), 1)
        self.assertIs(search_agent.tools[0], google_search)


class TestWeatherTools(unittest.TestCase):
    """The weather specialist's tools call the real APIs correctly."""

    def test_geocode_success(self):
        payload = {"status": "OK", "results": [{
            "formatted_address": "Denver, CO, USA",
            "geometry": {"location": {"lat": 39.7392, "lng": -104.9903}},
            "address_components": [{"types": ["country"], "short_name": "US"}]}]}
        with mock.patch("requests.get", return_value=_fake_response(payload=payload)):
            out = get_location_lat_long("Denver", "CO")
        self.assertEqual(out["status"], "success")
        self.assertAlmostEqual(out["latitude"], 39.7392)

    def test_geocode_rejects_non_us(self):
        payload = {"status": "OK", "results": [{
            "formatted_address": "London, UK",
            "geometry": {"location": {"lat": 51.5, "lng": -0.12}},
            "address_components": [{"types": ["country"], "short_name": "GB"}]}]}
        with mock.patch("requests.get", return_value=_fake_response(payload=payload)):
            out = get_location_lat_long("London", "England")
        self.assertEqual(out["status"], "error")
        self.assertIn("outside the United States", out["error_message"])

    def test_weather_two_hop_flow(self):
        points = {"properties": {"forecast": "https://api.weather.gov/x/forecast"}}
        forecast = {"properties": {"periods": [{
            "name": "This Afternoon", "temperature": 88, "temperatureUnit": "F",
            "shortForecast": "Sunny", "detailedForecast": "Sunny and warm."}]}}
        with mock.patch("requests.get", side_effect=[
                _fake_response(payload=points), _fake_response(payload=forecast)]) as m:
            out = get_current_weather(39.7392, -104.9903)
        self.assertEqual(m.call_count, 2)          # metadata hop, then forecast hop
        self.assertEqual(out["temperature"], 88)
        # Omitting User-Agent returns 403 from the NWS, so assert we send one.
        self.assertIn("User-Agent", m.call_args_list[0].kwargs["headers"])

    def test_weather_404_is_out_of_bounds(self):
        with mock.patch("requests.get", return_value=_fake_response(status_code=404)):
            out = get_current_weather(51.5, -0.12)
        self.assertEqual(out["status"], "error")
        self.assertIn("outside the United States", out["error_message"])


_suite = unittest.TestSuite([
    unittest.TestLoader().loadTestsFromTestCase(tc)
    for tc in (TestMultiAgentStructure, TestWeatherTools)])
unittest.TextTestRunner(verbosity=2).run(_suite)

test_root_exposes_both_specialists_as_tools (__main__.TestMultiAgentStructure.test_root_exposes_both_specialists_as_tools) ... ok
test_search_agent_uses_google_search (__main__.TestMultiAgentStructure.test_search_agent_uses_google_search) ... ok
test_specialists_are_tools_not_sub_agents (__main__.TestMultiAgentStructure.test_specialists_are_tools_not_sub_agents) ... ok
test_specialists_have_descriptions_for_routing (__main__.TestMultiAgentStructure.test_specialists_have_descriptions_for_routing) ... ok
test_three_agents_exist (__main__.TestMultiAgentStructure.test_three_agents_exist) ... ok
test_weather_agent_has_both_chaining_tools (__main__.TestMultiAgentStructure.test_weather_agent_has_both_chaining_tools) ... ok
test_geocode_rejects_non_us (__main__.TestWeatherTools.test_geocode_rejects_non_us) ... ok
test_geocode_success (__main__.TestWeatherTools.test_geocode_success) ... ok
test_weather_404_is_out_of_bounds (__main__.TestWeatherTools.test_weather_404_is_out_of_bounds) ... ok
tes

<unittest.runner.TextTestResult run=10 errors=0 failures=0>